# 365 Probabilidades — Dia #053
## Qual a probabilidade de a sorte ter decidido mais da sua carreira do que você admite?

**Tipo:** Comportamental
**Data de publicação:** 2026-08-05
**Ferramenta:** Python
**Decisão analisada:** Devo confiar só no meu talento, ou aumentar minha exposição a oportunidades?
**Hashtag:** #365Probabilidades #Dia053

---

### 📖 A História

Existe uma história que a gente conta sobre a própria carreira. Nela, cada
conquista é mérito: o talento, o esforço, as escolhas certas. É uma história
boa. E é quase sempre incompleta.

Porque a gente lembra das decisões que tomou e esquece das portas que
simplesmente estavam abertas quando passou. O chefe que apostou. A vaga que
surgiu na hora. O "não" que, no fim, foi sorte.

Eu já me peguei atribuindo a mim coisas que, olhando com honestidade, foram
tanto acaso quanto competência. Não porque eu não me esforcei. Mas porque o
esforço encontrou uma janela que eu não escolhi.

Admitir isso parece tirar mérito. Faz o oposto. Porque se a sorte pesa tanto,
a pergunta muda de figura: não é "como eu fui boa o suficiente?", é "como eu
me coloco na frente de mais janelas?".

---

### 📚 O Conceito: A Ilusão da Meritocracia

O talento é distribuído mais ou menos por igual. Inteligência, habilidade,
esforço: quase todo mundo se agrupa perto da média, e pouquíssimos ficam nos
extremos. É uma curva de sino, simétrica.

O sucesso, não. Riqueza, fama, poder: uns pouquíssimos concentram quase tudo,
e a imensa maioria fica com muito pouco. É uma lei de potência, torta.

Aqui está o problema lógico: se o sucesso viesse do talento, ele seria tão
simétrico quanto o talento. Não é. Essa distância entre uma curva de sino e
uma cauda pesada tem um nome, e o nome é sorte.

---

### 🧮 O Modelo

Reproduzo aqui o modelo de Pluchino, Biondo e Rapisarda: 1.000 pessoas com
talento sorteado numa curva normal, 40 anos de carreira, topando eventos de
sorte e azar ao longo do caminho. Talento serve para *agarrar* a sorte, mas
só se ela cruzar o seu caminho. Semente fixa (42), para rodar igual aqui.

E vou um passo além deles: isolo a **exposição** — a frequência com que cada
pessoa topa oportunidades — para medir se a variável que você *controla* pesa
mais no sucesso do que o talento que você *não escolhe*. É a pergunta que o
estudo original deixou em aberto.

**Fontes:**
- Pluchino, Biondo & Rapisarda (2018) — *Talent versus Luck: The Role of
  Randomness in Success and Failure* · Advances in Complex Systems · N=1.000
  agentes · Prêmio Ig Nobel de Economia 2022
- Denrell & Liu (2012) — *Top performers are not the most impressive when
  extreme performance indicates unreliability* · PNAS · âncora empírica


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("✅ Bibliotecas carregadas")


✅ Bibliotecas carregadas


In [6]:
# --- DADOS DA LITERATURA E PARÂMETROS DO MODELO ---
# Pluchino, Biondo & Rapisarda (2018), Advances in Complex Systems
# Modelo de simulação (agent-based). SEM fator ×0.80: não é survey, é modelo.

SEMENTE   = 42        # semente fixa: roda igual em qualquer máquina
N         = 1000      # número de pessoas simuladas
PASSOS    = 80        # 40 anos de carreira, em passos de 6 meses
MT, ST    = 0.6, 0.1  # talento ~ Normal(média 0.6, desvio 0.1), no intervalo [0,1]
CAP_INI   = 10.0      # todo mundo começa com o MESMO capital
P_SORTE   = 0.05      # prob. de topar um evento de SORTE a cada passo
P_AZAR    = 0.05      # prob. de topar um evento de AZAR a cada passo

# EXTENSÃO (o passo além de Pluchino): a EXPOSIÇÃO — a variável que VOCÊ controla,
# ou seja, com que frequência você se põe no caminho de uma oportunidade.
MED_EXP   = 0.05      # exposição média (mesma taxa do modelo base, para comparar)
DES_EXP   = 0.02      # o quanto a exposição varia entre as pessoas

# Denrell & Liu (2012, PNAS) — âncora empírica (não entra no cálculo):
# o topo extremo sinaliza sorte, não superioridade; o "segundo melhor"
# costuma ser mais habilidoso e mais confiável.

print("=" * 65)
print("  PARÂMETROS — MODELO TALENTO vs SORTE (Pluchino et al., 2018)")
print("=" * 65)
print(f"\n  Pessoas simuladas:          {N}")
print(f"  Carreira:                   {PASSOS} passos (~40 anos)")
print(f"  Talento:                    Normal(média {MT}, desvio {ST})")
print(f"  Capital inicial (todos):    {CAP_INI:.0f}")
print(f"  Prob. sorte / azar por passo: {P_SORTE} / {P_AZAR}")
print(f"  Fator ×0.80:                NÃO se aplica (modelo, não survey)")
print("=" * 65)


  PARÂMETROS — MODELO TALENTO vs SORTE (Pluchino et al., 2018)

  Pessoas simuladas:          1000
  Carreira:                   80 passos (~40 anos)
  Talento:                    Normal(média 0.6, desvio 0.1)
  Capital inicial (todos):    10
  Prob. sorte / azar por passo: 0.05 / 0.05
  Fator ×0.80:                NÃO se aplica (modelo, não survey)


In [7]:
# --- O MODELO — SIMULAÇÃO ---
rng = np.random.default_rng(SEMENTE)

# Talento: uma curva normal, simétrica. Quase todo mundo perto da média.
T = np.clip(rng.normal(MT, ST, N), 0.01, 1.0)

# Todos começam iguais.
C = np.full(N, CAP_INI, dtype=float)

# 40 anos de carreira: a cada passo, você pode topar sorte e/ou azar.
for _ in range(PASSOS):
    sorte  = rng.random(N) < P_SORTE
    azar   = rng.random(N) < P_AZAR
    # SORTE só vira ganho se você tem talento para agarrá-la (prob. = talento).
    agarrou = sorte & (rng.random(N) < T)
    C[agarrou] *= 2      # oportunidade aproveitada: capital dobra
    C[azar]    /= 2      # azar: capital pela metade

# --- RESULTADOS ---
i_vencedor  = np.argmax(C)
i_talentoso = np.argmax(T)
ordenado    = np.argsort(C)
top1pct     = ordenado[-N//100:]           # 1% mais ricos
mais_talentosos = np.where(T > 0.8)[0]      # os "gênios" (talento > 0.8)

# Concentração (Pareto)
Cs = np.sort(C)[::-1]
share_top20 = Cs[:N//5].sum() / Cs.sum()

# Assinatura: correlação entre talento e sucesso
r = np.corrcoef(T, np.log(C + 1e-9))[0, 1]
z = np.arctanh(r); se = 1/np.sqrt(N-3)
ic_r = np.tanh([z - 1.96*se, z + 1.96*se])
R2 = r**2

print("=" * 65)
print("  RESULTADO — QUEM CHEGOU AO TOPO?")
print("=" * 65)
print(f"\n  Talento médio da população:        {T.mean():.2f}")
print(f"  Talento do MAIS BEM-SUCEDIDO:      {T[i_vencedor]:.2f}   (capital {C[i_vencedor]:.0f})")
print(f"  Talento do MAIS TALENTOSO:         {T[i_talentoso]:.2f}   (capital {C[i_talentoso]:.0f})")
print(f"  → O vencedor é o mais talentoso?   {'SIM' if i_vencedor==i_talentoso else 'NÃO'}")
print(f"\n  Talento médio do 1% mais rico:     {T[top1pct].mean():.2f}  (mal acima da média)")
print(f"  Capital médio dos 'gênios' (T>0.8): {C[mais_talentosos].mean():.0f}  vs média geral {C.mean():.0f}")
print(f"\n  Concentração (Pareto):")
print(f"  → 20% mais ricos detêm {share_top20*100:.0f}% de todo o capital")
print(f"\n  ASSINATURA ESTATÍSTICA — correlação talento × sucesso:")
print(f"  → r = {r:.3f}   IC 95% [{ic_r[0]:.3f}, {ic_r[1]:.3f}]")
print(f"  → R² = {R2*100:.1f}%  →  talento explica ~{R2*100:.0f}% de quem vence.")
print(f"     Os outros ~{100-R2*100:.0f}% são sorte e acúmulo.")
print("=" * 65)

# ============================================================
#  EXTENSÃO — A ALAVANCA QUE É SUA: EXPOSIÇÃO vs TALENTO
#  (o passo a mais, além do modelo original de Pluchino)
# ============================================================
# Mesma mecânica, mas agora cada pessoa tem uma EXPOSIÇÃO própria: a
# frequência com que topa oportunidades. Talento não se escolhe; exposição,
# sim. A pergunta: qual das duas move mais o sucesso?
rng2 = np.random.default_rng(SEMENTE)
Texp = np.clip(rng2.normal(MT, ST, N), 0.01, 1.0)             # talento (não se escolhe)
Eexp = np.clip(rng2.normal(MED_EXP, DES_EXP, N), 0.01, 0.12)  # exposição (você controla)
Cexp = np.full(N, CAP_INI, dtype=float)
for _ in range(PASSOS):
    sorte   = rng2.random(N) < Eexp    # encontra oportunidade conforme SUA exposição
    azar    = rng2.random(N) < P_AZAR  # o azar bate igual em todo mundo
    agarrou = sorte & (rng2.random(N) < Texp)
    Cexp[agarrou] *= 2
    Cexp[azar]    /= 2

lc     = np.log(Cexp + 1e-9)
r_tal  = np.corrcoef(Texp, lc)[0, 1]
r_exp  = np.corrcoef(Eexp, lc)[0, 1]
R2_tal, R2_exp = r_tal**2, r_exp**2
alavanca = R2_exp / max(R2_tal, 1e-9)

print("\n" + "=" * 65)
print("  EXTENSÃO — EXPOSIÇÃO vs TALENTO (o passo a mais)")
print("=" * 65)
print(f"\n  Correlação TALENTO   × sucesso:  r = {r_tal:.3f}  (R² = {R2_tal*100:.1f}%)")
print(f"  Correlação EXPOSIÇÃO × sucesso:  r = {r_exp:.3f}  (R² = {R2_exp*100:.1f}%)")
print(f"\n  → A exposição pesa ~{alavanca:.0f}x mais que o talento.")
print(f"  → O que você controla (aparecer) move mais que o que")
print(f"     você não controla (o talento com que nasceu).")
print("=" * 65)


  RESULTADO — QUEM CHEGOU AO TOPO?

  Talento médio da população:        0.60
  Talento do MAIS BEM-SUCEDIDO:      0.58   (capital 640)
  Talento do MAIS TALENTOSO:         0.92   (capital 10)
  → O vencedor é o mais talentoso?   NÃO

  Talento médio do 1% mais rico:     0.65  (mal acima da média)
  Capital médio dos 'gênios' (T>0.8): 17  vs média geral 12

  Concentração (Pareto):
  → 20% mais ricos detêm 78% de todo o capital

  ASSINATURA ESTATÍSTICA — correlação talento × sucesso:
  → r = 0.165   IC 95% [0.104, 0.225]
  → R² = 2.7%  →  talento explica ~3% de quem vence.
     Os outros ~97% são sorte e acúmulo.

  EXTENSÃO — EXPOSIÇÃO vs TALENTO (o passo a mais)

  Correlação TALENTO   × sucesso:  r = 0.134  (R² = 1.8%)
  Correlação EXPOSIÇÃO × sucesso:  r = 0.348  (R² = 12.1%)

  → A exposição pesa ~7x mais que o talento.
  → O que você controla (aparecer) move mais que o que
     você não controla (o talento com que nasceu).


In [8]:
# --- VISUALIZAÇÃO — GRÁFICOS SEPARADOS (fundo branco, dpi=150) ---

VERDE    = '#2a8a82'
VERMELHO = '#c0392b'
DOURADO  = '#c9a14a'
CINZA    = '#6b6a64'

Cp = np.clip(C, 0.05, None)   # piso para escala log

# ── GRÁFICO 1 — Talento é simétrico; sucesso é uma cauda ──
fig1, (axa, axb) = plt.subplots(1, 2, figsize=(11, 5.5))
axa.hist(T, bins=30, color=VERDE, alpha=0.85)
axa.axvline(T.mean(), color=CINZA, linestyle='--', linewidth=2)
axa.set_title('O talento\n(curva de sino, simétrica)', fontsize=12)
axa.set_xlabel('Talento'); axa.set_ylabel('Nº de pessoas')

axb.hist(Cp, bins=np.logspace(np.log10(Cp.min()), np.log10(Cp.max()), 30),
         color=VERMELHO, alpha=0.85)
axb.set_xscale('log')
axb.set_title('O sucesso\n(lei de potência, torta)', fontsize=12)
axb.set_xlabel('Capital final (escala log)'); axb.set_ylabel('Nº de pessoas')

fig1.suptitle('Talento é distribuído por igual. Sucesso não é.', fontsize=14, y=1.02)
plt.figtext(0.5, -0.02,
            'Reprodução do modelo de Pluchino et al. (2018), N=1.000 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-053-grafico-01-distribuicoes.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 — O vencedor é mediano ──
fig2, ax2 = plt.subplots(figsize=(10, 6))
labels2 = ['Média da\npopulação', 'O 1% mais\nrico', 'O MAIS\nBEM-SUCEDIDO', 'O talento\nmáximo que\nexistia']
valores2 = [T.mean(), T[top1pct].mean(), T[i_vencedor], T.max()]
cores2 = [CINZA, VERDE, DOURADO, VERMELHO]
bars = ax2.bar(labels2, valores2, color=cores2, alpha=0.9, width=0.55)
ax2.set_ylim(0, 1.0)
ax2.set_ylabel('Talento (de 0 a 1)')
ax2.set_title('Quem Chegou ao Topo Não Era o Mais Talentoso\nO mais bem-sucedido tem talento praticamente na média',
              fontsize=13, pad=15)
for bar, v in zip(bars, valores2):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.02,
             f'{v:.2f}', ha='center', fontweight='bold', fontsize=15)
plt.figtext(0.5, 0.01,
            'Fonte: reprodução de Pluchino et al. (2018) | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-053-grafico-02-vencedor-mediano.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 — A ASSINATURA: talento quase não prevê sucesso ──
fig3, ax3 = plt.subplots(figsize=(10, 6))
ax3.scatter(T, Cp, s=14, color=CINZA, alpha=0.45)
ax3.set_yscale('log')
# reta de regressão em log
b, a = np.polyfit(T, np.log(Cp), 1)
xs = np.linspace(T.min(), T.max(), 100)
ax3.plot(xs, np.exp(a + b*xs), color=VERMELHO, linewidth=2.5,
         label=f'tendência (quase plana)')
ax3.scatter([T[i_vencedor]], [Cp[i_vencedor]], color=DOURADO, s=120,
            zorder=5, label='o mais bem-sucedido')
ax3.scatter([T[i_talentoso]], [Cp[i_talentoso]], color=VERDE, s=120,
            zorder=5, label='o mais talentoso')
ax3.set_xlabel('Talento')
ax3.set_ylabel('Capital final (escala log)')
ax3.set_title('A Assinatura Estatística — Talento Não Prevê Sucesso\n'
              f'r = {r:.2f}   ·   R² = {R2*100:.0f}%   ·   talento explica ~{R2*100:.0f}% de quem vence',
              fontsize=13, pad=15)
ax3.legend(loc='lower right', frameon=False, fontsize=9)
plt.figtext(0.5, 0.01,
            'Fonte: reprodução de Pluchino et al. (2018) — correlação talento × sucesso, IC 95% | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-053-grafico-03-assinatura.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ── GRÁFICO 4 — A ALAVANCA QUE É SUA (extensão além de Pluchino) ──
fig4, ax4 = plt.subplots(figsize=(10, 6))
labels4 = ['TALENTO\n(você não escolhe)', 'EXPOSIÇÃO\n(você controla)']
valores4 = [R2_tal*100, R2_exp*100]
bars4 = ax4.bar(labels4, valores4, color=[CINZA, DOURADO], alpha=0.9, width=0.45)
ax4.set_ylim(0, max(valores4)*1.28)
ax4.set_ylabel('Quanto explica do sucesso  (R², %)')
ax4.set_title('A Alavanca Que É Sua\nNo modelo, a exposição pesa cerca de %.0fx mais que o talento' % alavanca,
              fontsize=13, pad=15)
for bar, v in zip(bars4, valores4):
    ax4.text(bar.get_x() + bar.get_width()/2, v + max(valores4)*0.03,
             f'{v:.0f}%', ha='center', fontweight='bold', fontsize=18)
plt.figtext(0.5, 0.01,
            'Extensão além de Pluchino et al. (2018): exposição = frequência de topar oportunidades | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-053-grafico-04-alavanca.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")


✅ Gráfico 1 salvo!
✅ Gráfico 2 salvo!
✅ Gráfico 3 salvo!
✅ Gráfico 4 salvo!


### 💡 O Insight

**O mais bem-sucedido da simulação tinha talento abaixo da média. O mais
talentoso de todos terminou sem nada.**
**Talento explica cerca de 3% de quem chega ao topo. Os outros 97% são sorte
e acúmulo.**

Isso não quer dizer que talento não importe. No modelo, é o talento que te
deixa *agarrar* a sorte quando ela aparece. Mas você só agarra a sorte que
cruza o seu caminho. E quantas cruzam depende de quanto você se expõe.

Por isso o mais talentoso quase nunca vence. Não por falta de capacidade, mas
porque bater no jackpot exige uma sequência de golpes de sorte que nenhuma
quantidade de talento garante.

A leitura confortável seria "então é tudo sorte, não adianta". A honesta é o
contrário. Se a sorte decide quase tudo, a única alavanca que está na sua mão
é a exposição: quantas portas você bate, quantas salas você entra, quantas
vezes você se coloca no caminho do acaso.

E aqui entra o passo além do estudo original. Quando eu deixo cada pessoa ter
uma exposição própria e comparo, a exposição pesa cerca de **7 vezes mais** que
o talento sobre quem chega ao topo. Não é consolo: é a variável que você
controla vencendo a que você não controla.

Talento é o que você tem.
Exposição é o que você faz com a chance de ter sorte.

A pergunta que fica:

*Quantas portas você não bateu porque achou que talento bastava?*

---

### ⚠️ Limitações do Modelo

- Pluchino et al. (2018) é uma **simulação**, não a observação de carreiras reais. Ela demonstra uma lógica (talento não basta), não mede o mundo diretamente.
- A âncora empírica (Denrell & Liu, 2012) sustenta a direção do achado, mas também parte de um argumento estatístico, não de um experimento controlado.
- Os números exatos (talento do vencedor, r, concentração) dependem da semente e dos parâmetros; outras sementes dão valores próximos, com a mesma conclusão qualitativa.
- "Sorte" aqui é um evento aleatório abstrato; na vida real ela se mistura a privilégio, timing, rede de contatos e contexto, coisas que o modelo não separa.
- Na extensão, "exposição" é modelada como a taxa de topar oportunidades, então parte da força dela é estrutural. A comparação com o talento é justa (ambos são insumos com variação), mas o número (~7x) é uma propriedade do modelo estendido, não uma medida do mundo real.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
